<a href="https://colab.research.google.com/github/Yash-Yelave/Global_Gatway_RS/blob/main/queryable_recommendation_tables.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Environment Setup & Data Loading

In [1]:
import pandas as pd
import numpy as np
import sqlite3
import json
import io
import warnings
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from google.colab import files

warnings.filterwarnings("ignore")

# Prompt for file upload
print("Please upload your 'news_nlp_enriched.csv' from Day 5...")
uploaded = files.upload()

# Read the uploaded CSV
filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[filename]))

# Ensure we have a unique identifier for relational mapping
if 'article_id' not in df.columns:
    df['article_id'] = ['art_' + str(i).zfill(4) for i in range(len(df))]

print(f"\n✓ Successfully loaded {len(df)} articles.")

Please upload your 'news_nlp_enriched.csv' from Day 5...


Saving news_nlp_enriched.csv to news_nlp_enriched.csv

✓ Successfully loaded 250 articles.


The Trending Score Engine

In [2]:
# ==========================================
# 1. TRENDING SCORE LOGIC
# ==========================================
print("Calculating Trending Scores...")

# Simulate view counts for the sake of the recommendation engine (10 to 10,000 views)
# In production, this would be a live metric pulled from your analytics database
np.random.seed(42)
df['view_count'] = np.random.randint(10, 10000, size=len(df))

# Formula: log(views) * freshness_decay
# We use log scale so an article with 10,000 views doesn't completely eclipse a fresh article with 500 views.
df['trending_score'] = np.log1p(df['view_count']) * df['freshness_decay_score']

# Normalize the trending score to a clean 0.0 to 10.0 scale for the UI
max_score = df['trending_score'].max()
df['trending_score'] = (df['trending_score'] / max_score * 10).round(2)

# Create the specific Trending Table layout
df_trending = df[['article_id', 'title', 'auto_category', 'view_count', 'freshness_decay_score', 'trending_score']]
df_trending = df_trending.sort_values(by='trending_score', ascending=False)

print(f"✓ Trending table generated. Top article score: {df_trending['trending_score'].iloc[0]}")

Calculating Trending Scores...
✓ Trending table generated. Top article score: 10.0


Content Similarity Engine (TF-IDF)

In [3]:
# ==========================================
# 2. COMPUTE TF-IDF COSINE SIMILARITY
# ==========================================
print("Computing Article Similarity Matrix...")

# Combine title and description for a robust summary
text_corpus = df['title'].fillna("") + " " + df['description'].fillna("")

# Vectorize the text
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
tfidf_matrix = vectorizer.fit_transform(text_corpus)

# Compute the Cosine Similarity matrix
cosine_sim_matrix = cosine_similarity(tfidf_matrix, tfidf_matrix)

# Build the relational Similar Articles table
similar_articles_data = []

for idx in range(len(df)):
    art_id = df.iloc[idx]['article_id']

    # Get similarity scores for this specific article
    sim_scores = list(enumerate(cosine_sim_matrix[idx]))

    # Sort by similarity (descending), ignoring the first one (which is the article matching itself)
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:6]

    # Extract the article IDs and similarity scores of the top 5 matches
    top_matches = [{"similar_article_id": df.iloc[i[0]]['article_id'], "similarity_score": round(i[1], 4)} for i in sim_scores]

    similar_articles_data.append({
        "article_id": art_id,
        "similar_articles_json": json.dumps(top_matches) # Store as JSON array for easy DB retrieval
    })

df_similar = pd.DataFrame(similar_articles_data)
print("✓ Similar Articles mapped (Top 5 per article).")

Computing Article Similarity Matrix...
✓ Similar Articles mapped (Top 5 per article).


Category Grouping & Database Export

In [4]:
# ==========================================
# 3. CATEGORY & KEYWORD CLUSTERING
# ==========================================
print("Grouping Category Rankings...")

# Rank articles within their specific categories based on the new trending score
df['category_rank'] = df.groupby('auto_category')['trending_score'].rank(method='dense', ascending=False)

df_category_rankings = df[['article_id', 'title', 'auto_category', 'category_rank', 'trending_score']]
df_category_rankings = df_category_rankings.sort_values(by=['auto_category', 'category_rank'])

# ==========================================
# 4. DATABASE EXPORT
# ==========================================
db_filename = "recommendations.db"
conn = sqlite3.connect(db_filename)

print(f"\nWriting dedicated tables to {db_filename}...")

# 1. Main Master Table
df.to_sql("master_articles", conn, if_exists="replace", index=False)

# 2. Trending Table
df_trending.to_sql("trending_articles", conn, if_exists="replace", index=False)

# 3. Similar Articles Table
df_similar.to_sql("similar_articles", conn, if_exists="replace", index=False)

# 4. Category Rankings Table
df_category_rankings.to_sql("category_rankings", conn, if_exists="replace", index=False)

conn.commit()
conn.close()

print("\n" + "="*50)
print("DAY 6 PIPELINE COMPLETE!")
print("Ready for downstream API querying.")
print("="*50)

# Trigger the download in Colab
print("\nDownloading database file...")
files.download(db_filename)

Grouping Category Rankings...

Writing dedicated tables to recommendations.db...

DAY 6 PIPELINE COMPLETE!
Ready for downstream API querying.



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:
import pandas as pd
import numpy as np
import json
import io
import warnings
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from google.colab import files
from IPython.display import display, HTML

warnings.filterwarnings("ignore")

# ==========================================
# 0. UPLOAD DATA
# ==========================================
print("Please upload your 'news_nlp_enriched.csv' from Day 5...")
uploaded = files.upload()

filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[filename]))

# Ensure we have unique IDs
if 'article_id' not in df.columns:
    df['article_id'] = ['art_' + str(i).zfill(4) for i in range(len(df))]

print(f"\n✓ Successfully loaded {len(df)} articles.")

# ==========================================
# 1. TRENDING SCORE LOGIC
# ==========================================
print("\n1/3 Calculating Trending Scores...")

# Simulate views (since we don't have live user data yet)
np.random.seed(42)
df['view_count'] = np.random.randint(10, 10000, size=len(df))

# Formula: log(views) * freshness_decay
df['trending_score'] = np.log1p(df['view_count']) * df['freshness_decay_score']

# Normalize to a 0-10 scale
max_score = df['trending_score'].max()
df['trending_score'] = (df['trending_score'] / max_score * 10).round(2)

df_trending = df[['article_id', 'title', 'auto_category', 'view_count', 'freshness_decay_score', 'trending_score']]
df_trending = df_trending.sort_values(by='trending_score', ascending=False)

# ==========================================
# 2. COMPUTE TF-IDF COSINE SIMILARITY
# ==========================================
print("2/3 Computing Article Similarity Matrix...")

text_corpus = df['title'].fillna("") + " " + df['description'].fillna("")

vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
tfidf_matrix = vectorizer.fit_transform(text_corpus)
cosine_sim_matrix = cosine_similarity(tfidf_matrix, tfidf_matrix)

similar_articles_data = []

for idx in range(len(df)):
    art_id = df.iloc[idx]['article_id']
    sim_scores = list(enumerate(cosine_sim_matrix[idx]))

    # Sort descending, drop the first one (it's the article matching itself), keep top 5
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:6]

    top_matches = [{"similar_article_id": df.iloc[i[0]]['article_id'], "similarity_score": round(i[1], 4)} for i in sim_scores]

    similar_articles_data.append({
        "article_id": art_id,
        "similar_articles_json": json.dumps(top_matches)
    })

df_similar = pd.DataFrame(similar_articles_data)

# ==========================================
# 3. CATEGORY & KEYWORD CLUSTERING
# ==========================================
print("3/3 Grouping Category Rankings...")

df['category_rank'] = df.groupby('auto_category')['trending_score'].rank(method='dense', ascending=False)

df_category_rankings = df[['article_id', 'title', 'auto_category', 'category_rank', 'trending_score']]
df_category_rankings = df_category_rankings.sort_values(by=['auto_category', 'category_rank'])

# ==========================================
# 4. DISPLAY RESULTS IN COLAB
# ==========================================
print("\n" + "="*50)
print("DAY 6 PIPELINE COMPLETE! Here are your Recommendation Tables:")
print("="*50)

print("\n\n📊 TABLE 1: TRENDING ARTICLES (Top 5)")
display(df_trending.head(5))

print("\n\n🔗 TABLE 2: SIMILAR ARTICLES ('Read More' Widget) - First 5 rows")
display(df_similar.head(5))

print("\n\n📑 TABLE 3: CATEGORY RANKINGS (Top 5 across categories)")
display(df_category_rankings.head(5))

Please upload your 'news_nlp_enriched.csv' from Day 5...


Saving news_nlp_enriched.csv to news_nlp_enriched (2).csv

✓ Successfully loaded 250 articles.

1/3 Calculating Trending Scores...
2/3 Computing Article Similarity Matrix...
3/3 Grouping Category Rankings...

DAY 6 PIPELINE COMPLETE! Here are your Recommendation Tables:


📊 TABLE 1: TRENDING ARTICLES (Top 5)


,article_id,title,auto_category,view_count,freshness_decay_score,trending_score
9,4d83f4049ae2,flydubai to launch daily Bangkok flights from ...,technology,8332,1.0000,10.00
188,c537e1b7097f,AI swarms could hijack democracy without anyon...,technology,5801,1.0000,9.60
185,4c2415707a32,Scientists Find Cheaper Way to Kill Western Dr...,general,5259,1.0000,9.49
246,50db63baa17f,Michael Review: Jaafar Jackson Dazzles As His ...,general,1164,1.0000,7.82
141,a224803feccd,"The Download: turning down human noise, and LA...",technology,9924,0.7408,7.55




🔗 TABLE 2: SIMILAR ARTICLES ('Read More' Widget) - First 5 rows


,article_id,similar_articles_json
0,9b8c748a78a0,"[{""similar_article_id"": ""b19b51e256bd"", ""simil..."
1,6c85e93eb2d8,"[{""similar_article_id"": ""210e676f4ba6"", ""simil..."
2,8f53abce0bfc,"[{""similar_article_id"": ""b19b51e256bd"", ""simil..."
3,ff2d206b77d7,"[{""similar_article_id"": ""1db2b9da3430"", ""simil..."
4,7de36895b31a,"[{""similar_article_id"": ""3d71190da94e"", ""simil..."




📑 TABLE 3: CATEGORY RANKINGS (Top 5 across categories)


,article_id,title,auto_category,category_rank,trending_score
116,db8be5db4b3b,What s the key to better vegan cheese? Microbr...,business,1.0,7.48
39,82d055ba524d,hong kong property investment soars on lower f...,business,2.0,7.05
120,fdbe0d97ddf4,Blue Energy raises $380M to build grid-scale n...,business,3.0,7.01
144,337000087609,Colossal Biosciences said it cloned red wolves...,business,4.0,6.82
177,b1d652780252,crypto market strength led by bitcoin as altco...,business,5.0,6.74
